# Correlated Noise Simulation

This notebook demonstrates how to simulate **correlated (multi-qubit) noise** using the QDK simulators.
While the [noise.ipynb](noise.ipynb) notebook covers independent Pauli noise applied uniformly to all gates,
this notebook shows how to:

1. Configure per-gate noise using `NoiseConfig`.
2. Define custom noise intrinsics to model multi-qubit noise for custom gates.

First, make sure prerequisites are available. The package and extra `qdk[jupyter]` must be already installed.

In [ ]:
import qdk
from qdk import qsharp, openqasm
from qdk.simulation import NoiseConfig, run_qir
from qdk.widgets import Histogram

## 1. Per-Gate Noise with `NoiseConfig`

The convenience functions `DepolarizingNoise(p)` and `BitFlipNoise(p)` apply the same noise
to every gate. For more realistic modeling, you often need different error rates for different
gates, e.g., two-qubit gates are typically noisier than single-qubit gates, and they might apply
different noise to each qubit.

`NoiseConfig` gives you a noise table for each gate (H, X, CNOT, etc.), and each table
supports arbitrary Pauli error strings.

### Bell pair with per-gate noise

Define a Bell pair circuit in Q# and run it with different noise rates on single-qubit vs. two-qubit gates.

In [ ]:
%%qsharp

operation BellPair() : Result[] {
    use qs = Qubit[2];
    H(qs[0]);
    CNOT(qs[0], qs[1]);
    MResetEachZ(qs)
}

First, run without noise to establish a baseline. We expect only `[Zero, Zero]` and `[One, One]`.

In [ ]:
result = qsharp.run("BellPair()", 1000, seed=42)
Histogram(result)

Now configure `NoiseConfig` with different noise rates per gate:
- H gate: 0.5% depolarizing (single-qubit gates are relatively clean)
- CNOT (cx): 5% depolarizing (two-qubit gates are typically 10× noisier)
- Measurement (mresetz): 1% bit-flip (measurement readout error)

In [ ]:
noise = NoiseConfig()
noise.h.set_depolarizing(0.005)
noise.cx.set_depolarizing(0.05)
noise.mresetz.set_bitflip(0.01)

result = qsharp.run("BellPair()", 1000, noise=noise, seed=42)
Histogram(result)

The histogram now shows some `[Zero, One]` and `[One, Zero]` outcomes from errors.
Since the CNOT has 10× more noise than the H gate, most errors come from the two-qubit gate.

### Arbitrary Pauli errors on a gate

Each `NoiseTable` supports arbitrary Pauli strings. For example, on a CNOT gate we can
set specific correlated errors using multi-character Pauli strings. Each character in the
string corresponds to a qubit argument of the gate:

| Pauli string | Meaning |
|---|---|
| `xi` | X error on control, identity on target |
| `ix` | Identity on control, X error on target |
| `xx` | Correlated X error on both qubits simultaneously |
| `zz` | Correlated Z (dephasing) on both qubits — a common crosstalk error |

In [ ]:
# Model a CNOT gate with 5% correlated IX and 2% correlated ZI on both qubits.
noise = NoiseConfig()
noise.cx.ix = 0.05  # 5% probability of correlated IX error
noise.cx.zi = 0.02  # 5% probability of correlated ZI error

result = qsharp.run("BellPair()", 1000, noise=noise, seed=42)
Histogram(result)

You can also set multiple Pauli error channels at once using `set_pauli_noise`:

In [ ]:
noise = NoiseConfig()
noise.cx.set_pauli_noise([
    ("XI", 0.01),   # 1% X error on control only
    ("IX", 0.01),   # 1% X error on target only
    ("XX", 0.005),  # 0.5% correlated XX
    ("ZZ", 0.02),   # 2% correlated ZZ crosstalk
])

result = qsharp.run("BellPair()", 1000, noise=noise, seed=42)
Histogram(result)

## 2. Custom Noise Intrinsics

For more complex noise models, you can define custom noise intrinsics. These
are no-op gates in the ideal circuit that act as insertion points for correlated noise during
simulation. This is useful for modeling correlated noise on custom gates (those not included in `NoiseConfig`).

A noise intrinsic is declared with `@NoiseIntrinsic()` in Q# (or `@qdk.qir.noise_intrinsic`
in OpenQASM). The noise model is then configured in Python via `NoiseConfig.intrinsic()`.

### Q# example: modeling crosstalk between qubits

Imagine a 3-qubit register where applying a gate on qubit 0 causes crosstalk
(correlated ZZ dephasing) on qubits 1 and 2. We model this by inserting a noise
intrinsic after the gate.

In [ ]:
qdk.init(target_profile=qdk.TargetProfile.Adaptive_RIF)

In [ ]:
%%qsharp

// A noise intrinsic representing crosstalk on 3 qubits.
// In the ideal circuit this is a no-op; the simulator injects
// Pauli errors according to the NoiseConfig.
@NoiseIntrinsic()
operation Crosstalk3Q(q0: Qubit, q1: Qubit, q2: Qubit) : Unit {
    body intrinsic;
}

// Prepare a GHZ state on 3 qubits, with crosstalk after each CNOT.
operation GHZ() : Result[] {
    use qs = Qubit[3];
    H(qs[0]);
    CNOT(qs[0], qs[1]);
    Crosstalk3Q(qs[0], qs[1], qs[2]);  // crosstalk hits all 3 qubits
    CNOT(qs[1], qs[2]);
    Crosstalk3Q(qs[0], qs[1], qs[2]);  // crosstalk again
    MResetEachZ(qs)
}

Compile to QIR so we can run with a custom `NoiseConfig`:

In [ ]:
qir = qsharp.compile("GHZ()")

First, run without noise, we should see a clean GHZ state (`000` and `111` only):

In [ ]:
result = run_qir(qir, shots=1000, seed=42)
Histogram(result)

Now configure the crosstalk noise intrinsic. We'll model two types of correlated errors:
- `ixx`: correlated XX on qubits 1 and 2 (10% probability)
- `xxi`: correlated XX on qubits 0 and 1 (5% probability)

In [ ]:
noise = NoiseConfig()
table = noise.intrinsic("Crosstalk3Q", num_qubits=3)
table.ixx = 0.10  # 10% XX on qubits 1-2
table.xxi = 0.05  # 5% XX on qubits 0-1

result = run_qir(qir, shots=1000, noise=noise, seed=42)
Histogram(result)

The crosstalk breaks the perfect GHZ correlations, we now see outcomes like `001`, `010`,
etc. that would be impossible in the ideal circuit.

### OpenQASM example

The same noise intrinsic pattern works in OpenQASM using the `@qdk.qir.noise_intrinsic`
annotation on a custom gate definition.

In [ ]:
qasm_source = """
OPENQASM 3.0;
include "stdgates.inc";

// A noise intrinsic representing crosstalk on 3 qubits.
// In the ideal circuit this is a no-op; the simulator injects
// Pauli errors according to the NoiseConfig.
@qdk.qir.noise_intrinsic
gate crosstalk_3q q0, q1, q2 {}

qubit[3] qs;

// Prepare a GHZ state on 3 qubits, with crosstalk after each CNOT.
h qs[0];
cx qs[0], qs[1];
crosstalk_3q qs[0], qs[1], qs[2];  // crosstalk hits all 3 qubits
cx qs[1], qs[2];
crosstalk_3q qs[0], qs[1], qs[2];  // crosstalk again

bit[3] res = measure qs;
"""

qir_qasm = openqasm.compile(
    qasm_source,
    output_semantics=openqasm.OutputSemantics.OpenQasm,
    target_profile=qdk.TargetProfile.Base,
)

In [ ]:
noise = NoiseConfig()
table = noise.intrinsic("crosstalk_3q", num_qubits=3)
table.ixx = 0.10  # 10% XX on qubits 1-2
table.xxi = 0.05  # 5% XX on qubits 0-1

result = run_qir(qir_qasm, shots=1000, noise=noise, seed=42)
Histogram(result)